In [1]:
import pandas as pd
import numpy as np
import librosa
from tqdm import tqdm

df = pd.read_csv('data/track_genre_mapping_clean.csv')
print(df.shape)
df.head()

(7994, 5)


,track_id,genre,filepath,file_exists,filesize
0,2,Hip-Hop,data/fma_small\000\000002.mp3,True,960738
1,5,Hip-Hop,data/fma_small\000\000005.mp3,True,961580
2,10,Pop,data/fma_small\000\000010.mp3,True,721040
3,140,Folk,data/fma_small\000\000140.mp3,True,480497
4,141,Folk,data/fma_small\000\000141.mp3,True,480945


In [2]:
def extract_features(y, sr):
    features = {}
    
    # MFCC (13 koefisien, ambil mean & std tiap koefisien)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc{i+1}_mean'] = np.mean(mfcc[i])
        features[f'mfcc{i+1}_std'] = np.std(mfcc[i])
    
    # Chroma STFT
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features['chroma_mean'] = np.mean(chroma)
    features['chroma_std'] = np.std(chroma)
    
    # Spectral centroid
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    features['centroid_mean'] = np.mean(centroid)
    features['centroid_std'] = np.std(centroid)
    
    # Spectral bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features['bandwidth_mean'] = np.mean(bandwidth)
    features['bandwidth_std'] = np.std(bandwidth)
    
    # Spectral rolloff
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features['rolloff_mean'] = np.mean(rolloff)
    features['rolloff_std'] = np.std(rolloff)
    
    # Zero-crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)
    features['zcr_mean'] = np.mean(zcr)
    features['zcr_std'] = np.std(zcr)
    
    # RMS energy
    rms = librosa.feature.rms(y=y)
    features['rms_mean'] = np.mean(rms)
    features['rms_std'] = np.std(rms)
    
    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features['tempo'] = tempo
    
    return features

In [3]:
sample_path = df.loc[0, 'filepath']
y, sr = librosa.load(sample_path, sr=22050)
y = y / np.max(np.abs(y))  # normalisasi

clip = y[:3*sr]  # ambil klip 3 detik pertama
feat = extract_features(clip, sr)

print("Jumlah fitur:", len(feat))
feat

Jumlah fitur: 39


{'mfcc1_mean': np.float32(-71.327),
 'mfcc1_std': np.float32(51.64459),
 'mfcc2_mean': np.float32(62.290558),
 'mfcc2_std': np.float32(27.561106),
 'mfcc3_mean': np.float32(-13.772001),
 'mfcc3_std': np.float32(18.034132),
 'mfcc4_mean': np.float32(5.9821496),
 'mfcc4_std': np.float32(19.303858),
 'mfcc5_mean': np.float32(-7.7526402),
 'mfcc5_std': np.float32(18.31591),
 'mfcc6_mean': np.float32(7.903297),
 'mfcc6_std': np.float32(10.219876),
 'mfcc7_mean': np.float32(0.37286776),
 'mfcc7_std': np.float32(13.459387),
 'mfcc8_mean': np.float32(4.566412),
 'mfcc8_std': np.float32(9.481149),
 'mfcc9_mean': np.float32(-3.6069486),
 'mfcc9_std': np.float32(10.169502),
 'mfcc10_mean': np.float32(-0.7610783),
 'mfcc10_std': np.float32(6.911729),
 'mfcc11_mean': np.float32(-11.544494),
 'mfcc11_std': np.float32(9.4896145),
 'mfcc12_mean': np.float32(4.198539),
 'mfcc12_std': np.float32(14.178994),
 'mfcc13_mean': np.float32(-8.0524435),
 'mfcc13_std': np.float32(10.669559),
 'chroma_mean': np.

In [4]:
import os

output_file = 'data/features.csv'
checkpoint_every = 200  # simpan progress tiap 200 track

all_features = []
failed_extraction = []

# kalau sudah pernah jalan sebagian, lanjutkan dari situ
processed_ids = set()
if os.path.exists(output_file):
    existing = pd.read_csv(output_file)
    processed_ids = set(existing['track_id'].unique())
    all_features = existing.to_dict('records')
    print(f"Melanjutkan dari checkpoint, {len(processed_ids)} track sudah diproses.")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    if row['track_id'] in processed_ids:
        continue  # skip yang sudah diproses
    
    try:
        y, sr = librosa.load(row['filepath'], sr=22050)
        y = y / np.max(np.abs(y)) if np.max(np.abs(y)) > 0 else y
        
        clip_length = 3 * sr
        n_clips = len(y) // clip_length
        
        for i in range(n_clips):
            clip = y[i*clip_length : (i+1)*clip_length]
            feat = extract_features(clip, sr)
            feat['track_id'] = row['track_id']
            feat['genre'] = row['genre']
            feat['clip_idx'] = i
            all_features.append(feat)
        
    except Exception as e:
        failed_extraction.append(row['filepath'])
        continue
    
    # checkpoint: simpan progress berkala
    if idx % checkpoint_every == 0 and idx > 0:
        pd.DataFrame(all_features).to_csv(output_file, index=False)

# simpan final
pd.DataFrame(all_features).to_csv(output_file, index=False)
print("Selesai! Total klip terekstrak:", len(all_features))
print("Gagal ekstraksi:", len(failed_extraction))

  9%|▊         | 682/7994 [14:32<3:03:40,  1.51s/it]C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 10%|█         | 808/7994 [17:36<2:58:08,  1.49s/it]C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 15%|█▍        | 1191/7994 [26:51<3:42:27,  1.96s/it]C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 19%|█▊        | 1494/7994 [36:09<3:54:08,  2.16s/it]C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 20%|██        | 1622/7994 [40:35<4:01:20,  2.27s/it]C:\Users\ASUS\AppData\Roaming\Pyt

Selesai! Total klip terekstrak: 75296
Gagal ekstraksi: 0


In [5]:
features_df = pd.read_csv('data/features.csv')
print(features_df.shape)
features_df.head()

(75296, 42)


,mfcc1_mean,mfcc1_std,mfcc2_mean,mfcc2_std,mfcc3_mean,mfcc3_std,mfcc4_mean,mfcc4_std,mfcc5_mean,mfcc5_std,...,rolloff_mean,rolloff_std,zcr_mean,zcr_std,rms_mean,rms_std,tempo,track_id,genre,clip_idx
0,-71.327000,51.644590,62.290558,27.561106,-13.772001,18.034132,5.982150,19.303858,-7.752640,18.315910,...,6480.914401,1419.826926,0.157493,0.070559,0.129664,0.058296,[161.49902344],2,Hip-Hop,0
1,-78.129620,59.486626,54.927100,35.565380,-9.215000,24.406134,5.029922,16.306326,-4.937216,15.385864,...,6585.681716,1500.137366,0.190009,0.119099,0.128513,0.080900,[161.49902344],2,Hip-Hop,1
2,-66.667960,48.368900,63.013588,27.507763,-13.263946,19.643131,12.383762,14.758546,-1.616956,17.355505,...,6183.756197,1467.140103,0.160209,0.089279,0.141160,0.084941,[83.35433468],2,Hip-Hop,2
3,-79.796700,56.479572,60.561516,30.618351,-3.802702,24.794542,9.478866,14.462512,-5.341378,13.317841,...,6550.069111,1471.926072,0.160660,0.092230,0.142296,0.087260,[161.49902344],2,Hip-Hop,3
4,-54.840702,52.832400,65.101570,28.843266,-9.944346,23.935583,11.885717,20.594007,-0.393693,17.019888,...,6208.188101,1523.189132,0.163285,0.099907,0.165028,0.078045,[161.49902344],2,Hip-Hop,4


In [6]:
# cek tidak ada NaN
print("Jumlah NaN per kolom:")
features_df.isna().sum().sum()

Jumlah NaN per kolom:


np.int64(0)

In [7]:
# distribusi genre di level klip (bukan track)
features_df['genre'].value_counts()

genre
Pop              9443
Folk             9435
International    9428
Electronic       9415
Rock             9412
Experimental     9403
Instrumental     9390
Hip-Hop          9370
Name: count, dtype: int64